In [0]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("linreg_infections").getOrCreate()

df = spark.read.csv("/Workspace/Users/jessica.vargas006@mymdc.net/CAP4784 | Big Data/linreg.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+------+----------+
|   pop|infections|
+------+----------+
| 25101|       245|
| 61912|       215|
| 33341|      2076|
|409061|      5023|
|  7481|       189|
+------+----------+
only showing top 5 rows
root
 |-- pop: integer (nullable = true)
 |-- infections: integer (nullable = true)



In [0]:
df.columns

['pop', 'infections']

### 1) What is the explanatory variable(features)? Explain.
The explanatory variable is **`pop`** (population).  
It is the independent/input variable we use to predict the output. In Spark MLlib, `pop` is assembled into a vector column called **`features`** so the model can train.


In [0]:
display(df.select("pop", "infections"))

pop,infections
25101,245
61912,215
33341,2076
409061,5023
7481,189
18675,195
25581,123
22286,116
459598,3298
3915,430


### 2) What is the purpose of building this model?
The purpose is to **predict the number of infections (`infections`) from population (`pop`)** and to measure how infections change as population increases. This gives a simple baseline model and a numeric relationship (slope/intercept) we can use for estimation.


In [0]:
assembler = VectorAssembler(inputCols=["pop"], outputCol="features")
lr = LinearRegression(featuresCol="features", labelCol="infections")

pipeline = Pipeline(stages=[assembler, lr])

pipeline.getStages()


[VectorAssembler_f62307c5ea3a, LinearRegression_b4a13d636c4b]

### 3) Two stages of the pipeline
1. **VectorAssembler**: turns `pop` into the required vector column **`features`**.  
2. **LinearRegression**: trains the regression model to predict **`infections`** from **`features`**.


In [0]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train)
pred = model.transform(test)

pred.select("pop", "infections", "prediction").show(10)


+------+----------+------------------+
|   pop|infections|        prediction|
+------+----------+------------------+
| 20679|       120|   843.57793773006|
| 25581|       123| 859.5530550779844|
| 31459|        52| 878.7088567448754|
| 67197|       502| 995.1753487440848|
|153920|       234| 1277.796748532753|
|162812|       279|1306.7748683731743|
|251417|       290|1595.5295139490304|
|409061|      5023| 2109.275209419492|
+------+----------+------------------+



In [0]:
rmse_eval = RegressionEvaluator(labelCol="infections", predictionCol="prediction", metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol="infections", predictionCol="prediction", metricName="r2")

rmse = rmse_eval.evaluate(pred)
r2   = r2_eval.evaluate(pred)

rmse, r2


(1338.505565578928, 0.2921542436727691)

### Is the model making good predictions? Explain.
Not really — the model is **not making good predictions**.

- **RMSE:** 1338.51  
- **R²:** 0.292  

An **R² of ~0.29** means population (`pop`) only explains about **29%** of the variation in infections, so a lot of what drives infections is missing from the model. The **RMSE is large**, and you can see it in the sample predictions: for example, some rows with infections around **120–500** are being predicted closer to **800–2000+**, which is a big error.

This makes sense because we’re using only one predictor (`pop`). In real life, infections are likely affected by other factors (region, income, etc.), so with only population the model captures only a rough trend, not accurate predictions.
